# Team D - Classification using GNN

## Team Members

* Fabian Bäumer, 3734495
* Gerrit Erdt, 2299114

## Data and Example Download

* The data to use are described in the MAGIC-Projects.ipynb notebook 


## TASK 

* Improve your understanding of the data 
* use the pixel information features "clean_image"
    * they contain information of all the pixels for a simulated particle
        * "clean_image" contains the data after some some standard cleaning algorithms were applied. These data are used in the MAGIC analysis to derive the Hillas Parameters.
        * use the data of the both telescopes separately and together (as in stereo mode)
* Evaluate GNN architectures to create a model for classification
    * The GNNs can the the images as set of node and edge using locality features 
    * it is recommended pytorch-geometric    
    * search for other approaches in literature
    * think about new approaches
    * stay close with Jarred 
* [Optional]: Evaluate GNN architectures to create a model to predict hillas parameters 
    * see above
* compare the different approaches 





## Implementation

* Start here with your implementation and documentation




## Python runtime environment

The development has been done using Python 3.12.10. 
Pytorch is required, matching the installed CUDA version.
Additional packages/dependencies can be found in the requirements.txt file. 
torch-scatter is not strictly necessary, but in case it is missing, a warning will be generated. In case installation is desired, it needs to match the installed cuda/pytorch versions. 

## Folder structure and files

The implementation has been split in different folders/modules: 
```
src/                        - source root folder
├── main.py                 - main entry point, contains the calls to the training functions
├── data_loading/           - data loading modules
│   └── data_loading.py     - contains the DataLoader class and related functions
├── helper_functions/       - helper function modules
│   └── helper_functions.py - functions for data visualization and random seeding
├── magicdl/                - magicdl module provided by Jarred
└── models/                 - contains the model definitions and training routines
    ├── baseline_model.py   - implementation of a simple baseline GNN
    ├── focal_loss.py       - implementation of the used focal loss loss function
    └── model.py            - our final model implementation
```

## Visualization of training data
To display the data, a dataloader is required. For this example, a small subset (100 images) is sufficient. 
Since the main interest of the project is the classification of MAGIC cleaned images using GNNs, we are going to use the implemented data loader class directly, which provides us with a visualization of the graphs we are using for the classification task. 

Example images can be seen here: 

In [ ]:
import src.data_loading.data_loading as dl
import src.helper_functions.helper_functions as hf

hf.set_all_seeds() # set seeds for all further used libraries

visualization_loader, _, _, _ = dl.get_stereo_clean_dataset(100, batch_size=1)

hf.show_many_graphs()

In each diagram, the following information can be seen: 
- Graph for m1 on the left side
- Graph for m2 in the middle
- Graph of the difference (m1 - m2) on the right side. This is sometime useful to differentiate between weak signals from both telescopes when they align very closely. 

### Preprocessing applied to the data
Note that the visualization does not show the linear pixel values but instead the values we will be using during training. 
We found that it is much more efficient to use sqrt(m1) and sqrt(m2) as input features for the training. This can be explained by the high dynamic range of the pixels. Pixel values can range from 0 to 750 in the images. However, for the classification of gamma and proton events, it is much more important to get a grasp of the general shape of the event instead of single individual high values in the event. 
By applying a square root transformation, the dynamic range is compressed prior to the GNN, giving less attention to single bright pixels. This significantly improved the training process. 

A better demonstration of the effectiveness can be seen here: 
![Feature importance](feature_importance_four.png "Feature importance")

In this image, the first two images from the left side show the model loss (which should aproach 0.0) and the AUROC (which is the area under the ROC curve and should aproach 1.0). The image on the right side shows the effect, which the shuffling of all samples of feature n inside the test set has on the model performance (measured by the AUROC metric). Higher values indicate a higher importance for the model output. 
In this graph, F0 and F1 are the images from m1 and m2 respectively while F2 and F3 are sqrt(m1) and sqrt(m2) respectively. 
It is evident that the importance of F0 and F1 is very small, in absolute terms and in relative terms, in relation to F2 and F3. 
This supports our earlier conclusion that it is more efficient to train a model on sqrt(m1) and sqrt(m2) instead. 

Note also that the graphs are already sparse, meaning they only contain information from pixels where the intensity is > 0. This significantly speeds up all further computations. 

Additionally, the data has already been normalized using a min/max scaler. 
The min/max scaling is performed over the whole training dataset of m1 and m2 cleaned images combined, meaning that the same scaling is applied to images from both telescopes. It is applied after the square-root-transformation has been performed. 
During the development, we talked about the data normalization with Jarred. Specifically, we were interested in whether there are differences in the two telescopes, since they are not completely identical. According to Jarred, there are differences, but they are being taken care of by the data calibration and cleaning process which is applied to generate the cleaned images we are working with. 
However, to be sure, he suggested to verify this using histograms of a large number of samples from both telescopes. The results of this can be seen in the following graphs, showing the histogram for the first 50k images form both telescopes for each class: 

In [ ]:
hf.show_histograms_for_telescopes()

As can be seen in the diagrams, the differences between both telescopes after the cleaning process are insignificant. This justifies the applied process of normalizing them with a global set of parameters. 

## Goal definition
According to Jarred, current models (CNN-based approaches) achieve a correct classification in >= 95% of all images. 

For our purpose, we consider the following thresholds to define our success: 
- AUROC <= 0.75 for all test data: the model is completely unsuitable
- AUROC >= 0.75 and <= 0.95 for all test data: the model is generally suited for the task, but there seems to be room for improvement
- AUROC >= 0.95 for all test data: the model is on par with or surpasses the current state of the art models

## Baseline model
As a first test, a simple baseline GNN will be trained on the dataset to analyze how effective or ineffective a simple model is at classifying the data. 

### Baseline model architecture
The model begins with  2 Layers of graph convolutional networks, which increase the dimensionality from 2 to 64 and propagates information from each node (pixel) through each of its connected neighbours. 

The GNN is followed by a simple 2 layer MLP, using a sigmoid activation function and a stepwise reduction of the dimensionality to one output node. 

In total, the model has about 6k trainable parameters. 

### Baseline model training
The training will be done on the full dataset for 10 epochs, using a 70/15/15 split for train/val/test data. 
The results can be seen here: 

In [ ]:
import src.main as main_module

main_module.just_train_baseline()

As can be seen from the learning curves, the model does not over fit but performs very poorly in general. An AUROC value of 0.61 is considered inadequate as per our own goal definitions. 

This either indicates that the concept of using GNNs is unsuitable in general or that the model architecture is too simple and the capacity too low. 
Since Jarred had far better results using GNNs, we can assume that the actual reason is the later. 

## Improved model

Since the baseline model seemingly had an inadequate architecture or capacity, we tried to constructed a better model. 

### Improved model architecture 
The model begins with a 2 layer MLP which gets two values (sqrt(m1), sqrt(m2)) as input features. 
The purpose of the input net is to scale these two features up to a higher dimensionality to allow the following layers to extract more than two different layers of features from these. 
In this MLP, the input is scaled up to internal_dimensions (=64). An important step is the application of layer normalization, which normalizes each graph independently, thus creating a uniform input value for each layer of the neural network, regardless of the scale of the actual image. 
Additionally, we use GELU as activation function. GELU solves the problem of the "dying RELU" and avoids a sudden change in the output value for input values close to 0.0, thus enabling a more stable training especially for deeper neural networks. 

The RELU and GELU activation functions can be seen here: 



![RELU vs GELU](activation.png "RELU vs GELU")

(source: https://www.researchgate.net/figure/Comparison-of-the-ReLu-and-GeLu-activation-functions-ReLu-is-simpler-to-compute-but_fig3_370116538)

In conclusion, we chose GELU over RELU not directly because it performs better, but because of a lower probability for actually experiencing the worst case scenario in some neurons. 

We use the same activation and normalization function for the rest of the network. 

Additionally, in the input net, dropout is applied to improve generalization. 
We call the output of the input net "input_net_output". 

After the input net, which increases the dimensionality of the input data, the actual GNN follows. 

The GNN works by using multiple convolution steps, which allow information to propagate through the graph. Each convolution step uses a 2-layer MLP to process the information. 
This MLP follows the same basic structure as the input net (using GELU, Layer Normalization and Dropout). However, GNNs in PyTorch work by concatenating the input features with the differences to all its neighbours, so the input dimensionality of the GNN is 2*internal_dimensions, which is reduced to internal_dimensions inside the GNN convolution step. 
After the convolution step, an aggregation of max and mean features is applied, again doubling the dimensionality, which is again halved by a simple 1-layer MLP.
We call the output of the GNN "gnn_output".

The final step in the neural network is a classifier network.
This is a 4-layer MLP which follows the same basic structure again. 
In each layer, the dimensionality is divided by four to achieve a stepwise reduction in information. 
The output of this neural network is one dimensional. It represents the final classification output. 

Note that each major neural network (except for the one tasked with the re-projection inside the GNN) has an individual dropout rate to allow for a finer control over how much each of it is overfitting, since they have wildly different tasks and complexities. 

Note also that all of these in total four different neural networks are connected into one single large neural network, at least from an outside perspective. 

During a forward pass, some additional measures were taken to further improve learning and robustness of the network. 

The input is passed through the input_net to get the input_net_output. 
Using the input_net_output, the max and mean aggregations of it are computed and stored for later. 

Afterwards, the input_net_output is passed through each GNN step. 
In each GNN step, the input of this individual step is saved. 
Subsequently, the GNN step is applied and the input of this individual step is added to the output. This is used as the input for the next GNN step. 
The process of adding the input to the output is called residual connection. It prevents over smoothing (each neighbour passes its information to all of its neighbours for too many operations, thus smoothing the graph too much) and vanishing gradients, since there is a direct connection between all steps. 

Using the gnn_output, the mean and max aggregations are computed again. 

The mean and max aggregations of the input_net_output and of the gnn_output are concatenated and passed into the classifier to compute the final output (classification). 
The process of combining information from earlier stages of the network with the end of it is called jumping knowledge. It allows a direct path for information from one place to a much later one, thus increasing the amount of abstraction the net can do while still retaining small-scale features from the input and works as yet another feature against over smoothing. 

The model architecture is visualized in the following image: 

<img src="model_architecture.png" alt="Model architecture" title="Model architecture" width="500">

This network architecture allows for a high degree of flexibility and enables the network to use a diverse set of features for the classification task, focusing on robustness. It represents a clear upgrade in comparison to the baseline model. 

### Improved model training procedure
Additionally, significant improvements were applied to the training procedure. 

During data loading, the class distribution is automatically calculated. This distribution is used for the Focal Loss loss function we have implemented. This loss function offers us two significant improvements:
- Ability to use the class weighting to compensate for the in-equal class distribution
- Ability to weight elements differently depending on how 'certain' the classification is. This introduces a stronger punishment for more uncertain samples

In combination, this loss function is more robust than typical loss functions. 

Additionally, the optimizer has been changed to AdamW instead of Adam. Adam does not converge optimally when using L2 regularization, which is handled differently in AdamW. This leads to better and more efficient convergence during training. (source: https://optimization.cbe.cornell.edu/index.php?title=AdamW)

Instead of relying on a constant learning rate, a learning rate scheduler has been introduced. We opted for OneCycleLR, which starts with a low learning rate, increases it to a maximum learning rate and then decreases it again. It adapts to the number of planned training epochs and the number of batches per epoch, which makes it especially efficient for our HP optimization. 
Additionally, it increases the chance of hitting a global instead of a local optimum by first increasing and then decreasing the learning rate. This changes the typical HP of the learning rate to the maximum learning rate. 
In combination with the LR scheduler, gradient clipping has been added to minimize the risk of exploding gradients early in the training. This proved as a robust combination. 

For improved performance, we implemented automatic mixed precision, where PyTorch internally decides whether to use 16bit or 32bit floating point operations. This speeds up the training process, while sacrificing a small amount of accuracy, which is a trade off we were willing to accept for the ability to optimize over a larger parameter range. 

Also, early stopping was implemented. In case the validation loss does not improve for n subsequent epochs, the training is stopped and the best weights are restored. Due to the ramping up period of the LR scheduler and a minimum required amount of epochs, this does only apply after a minimum of m epochs. 

### Improved model training
The training process is twofold. 

In a first step, a 70/15/15 train/val/test split is created. 
From the 70% training data, 35% are used for the hyper parameter optimization, using a 60/40 split for training and validation data. 

Using the optuna framework, we can efficiently optimize the HP for our model and training routine. 

An initial set of HPs was defined, including: 
- Dropout in the input net
- Number of GNN layers applied
- Dropout in each GNN step
- Classifier dropout
- Number of internal dimensions
- Activation function
- Aggregated features
- Maximum learning rate
- L2 regularization strength
- Focal loss alpha
- Focal loss gamma

After long tests with optuna, the parameter importance reported by optuna was used to determine the most important HPs and choose more restrictive ranges for them. After a successfull optimization, optuna reports how much of the variance in the validation loss can be explained by each HP, allowing us to remove less important HPs and set them to the optimal values that were found earlier or at least reduce their range to save computation resources in later trials.

The reduced set of HPs for our model is: 
- Dropout in the input net
- Number of GNN layers applied
- Dropout in each GNN step
- Classifier dropout
- Maximum learning rate
- L2 regularization strength
- Focal loss alpha
- Focal loss gamma

In addition to reducing the HPs, we also used small trial-runs to reduce the range of parameters to a closer (bust still large enough) range around values we found to be useful. 

Of course, there are more HPs to be considered, but the list we compiled contains the most important ones as far as we could tell. In practice, it is impossible for us due to computational restraints to perform meaningful HPO for all possible HPs of the model and training process, so we had to narrow it down to more important ones, focusing on model robustness and computational efficiency. Although imperfect, this is a necessary trade off we have to do at some point. 

After the HP optimization, we know a close-to-optimal set of HPs for the HP optimization dataset. 

Using these HP values, the final training is performed using the full dataset. 

Since the full training dataset is larger, and we don't want to risk overfitting, the number of epochs is automatically calculated based on the ratio of the sizes of the training data sets for the final training and the HPO. That way, we get a higher number of total steps for the optimizer (meaning weight adaptions), which is compensated for by a larger number of training samples. With this methodology, we hope to improve model performance up to a reasonable degree while avoiding overfitting as good as possible. 

Final training results and predictions can be seen in the following graphs: 

<!-- ![Training results](best_so_far.jpeg "Training results") # TODO: Update -->

In [ ]:
import src.main as main_module

main_module.main() # performs HPO and final training, then shows the results and predictions


As can be seen in the learning graph, we achieved our highest goal of AUROC >= 0.95 for the test dataset. 
This means that the model is clearly on par with the reference values provided by Jarred, which we consider to be a success. 

Since the model continues to learn until the last epoch (as can be seen by a still decreasing validation loss) we have reasons to hope that the model could perform even better with more data or computation power to allow for better HPO. 
A problem we still encountered during training are hints for overfitting, since the validation curve is not clearly better than the learning curve. This may require a stronger regularization or could also be solved by simply having more data. 